# ClickHouse

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/clickhouse/clickhouse.ipynb)

ClickHouse is a column-oriented OLAP database for real-time analytics over append-only event data. ClickHouse DDL and system queries are shown as fenced SQL; the runnable cells use DuckDB (another in-process columnar engine) and small Python models to demonstrate the mechanisms: sorted parts and the sparse index, deduplication by version, incremental aggregation states, codecs, and querying Parquet in place.

*Last reviewed: November 2026, against ClickHouse 26.10 and the ClickHouse Cloud documentation as of November 2026.*

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is ClickHouse and how is it architected?

ClickHouse is a column-oriented OLAP database built for real-time analytics. It is shared-nothing: each node has local disks and its own CPU, and there is no single leader in the data path. A query can be sent to any node, and a `Distributed` table fans it out to shards.

+ **Table engine** - storage and behavior are chosen per table. `MergeTree` and its variants are the foundation; `Distributed` and `Replicated*` add scale and durability.
+ **Parts and partitions** - inserts create immutable parts; background merges combine them inside a partition.
+ **ClickHouse Keeper** - a Raft-based coordination service that replaces ZooKeeper for replication and distributed DDL.
+ **Clusters** - shards (horizontal data split) and replicas (copies of a shard), defined in `remote_servers` with per-node `macros`.

ClickHouse favors wide, append-only, denormalized tables with high insert throughput. It is not an OLTP database: frequent single-row updates stay expensive (lightweight `UPDATE` and `DELETE`, covered under operations, make occasional corrections cheap), there are no foreign keys, and joins are a secondary operation rather than the primary access path.

A query arrives at any node. That node parses it, plans it, and splits the work into blocks of columns. Each block flows through a pipeline of operators, and the CPU works on vectors of values rather than one row at a time. This vectorized execution is why one node can scan billions of rows per second.

Data is stored on disk as one file per column per part. A query that names five columns touches only five column files, so a wide table stays cheap when queries are selective. Nothing is read for a column the query does not mention.

**ClickHouse Cloud and SharedMergeTree.** Everything above describes self-managed, shared-nothing ClickHouse. ClickHouse Cloud separates storage from compute instead. The SharedMergeTree engine family keeps parts in object storage (S3, GCS or Azure Blob) and metadata in ClickHouse Keeper. Replicas never talk to each other, and a table can have hundreds of replicas without any shards. The DDL stays the same: a `MergeTree` or `ReplacingMergeTree` table is created as its Shared counterpart automatically.

With no shards to fan out to, **parallel replicas** spread one query across replicas instead. A coordinator hands granule ranges to each replica and merges the partial results. Turn it on with `enable_parallel_replicas = 1` and `max_parallel_replicas`. It is skipped for `FINAL` queries, ignores projections, and can be slower for small queries or high-cardinality aggregations, so measure before enabling it globally. Recent releases can also decide per query through `automatic_parallel_replicas_mode`.

## How does MergeTree store parts, partitions, and ORDER BY?

`MergeTree` sorts rows by the `ORDER BY` expression, splits data into partitions, writes each insert as a part, and merges parts in the background. The `PRIMARY KEY` is the sparse index prefix and, when omitted, defaults to `ORDER BY`.

```sql
CREATE TABLE analytics.events
(
    event_time  DateTime,
    user_id     UInt64,
    country     LowCardinality(String),
    event_type  LowCardinality(String),
    amount      Decimal(18, 2)
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(event_time)
ORDER BY (country, event_time, user_id)
SETTINGS index_granularity = 8192;

INSERT INTO analytics.events VALUES
    (now(), 101, 'DE', 'purchase', 42.50),
    (now(), 102, 'FR', 'view', 0);
```

+ `ORDER BY` should lead with the lowest-cardinality column you filter on, then the timestamp, then the high-cardinality key. Sorted data is what makes range scans fast.
+ Keep monthly or daily partitions for time-series data; too many tiny partitions hurt merges and metadata.
+ `PRIMARY KEY` must be a prefix of `ORDER BY`. Set it explicitly only when you want a shorter sparse index than the sort key.
+ A part is the unit of merging and each insert creates one part, so insert less often with more rows per insert.

An insert writes one or more immutable parts. A part covers exactly one partition and holds that insert's rows sorted by the `ORDER BY` key. Background merges combine several parts of the same partition into one larger part, and merges never cross partitions.

Every part is split into granules of `index_granularity` rows (8192 by default). Each granule gets one entry, a mark, in the primary index, and one mark per column file that points at the granule's byte offset. The index therefore holds roughly `rows / index_granularity` entries, which stays small enough to keep in memory.

Parts per partition is the read-side health metric. Each part means more files to open and more merge work to schedule. Aim for a few hundred active parts per partition at most. The server delays inserts when a partition passes `parts_to_delay_insert` (default 1000) and rejects them above `parts_to_throw_insert` (default 3000).

`ORDER BY` defines the physical sort order. `PRIMARY KEY` must be a prefix of it and defaults to the full `ORDER BY`. Set `PRIMARY KEY` shorter when you want a smaller index in memory but still want the longer sort key for range reads.

```sql
-- Long sort key for ordered scans, short index to keep memory low
CREATE TABLE analytics.events_pk
(
    event_date Date,
    country    LowCardinality(String),
    user_id    UInt64,
    event_time DateTime
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(event_date)
ORDER BY (country, event_date, user_id, event_time)
PRIMARY KEY (country, event_date);

-- Read parts and size per partition
SELECT partition, name, rows,
       formatReadableSize(bytes_on_disk) AS size,
       marks
FROM system.parts
WHERE table = 'events_pk' AND active
ORDER BY partition, name
LIMIT 20;
```

Read the row count of the result as the part count per partition. A partition with thousands of parts is a merge backlog, not a storage problem.

Common mistake: putting the most selective column first. A query that filters time alone cannot use an index that leads with `user_id`, so it reads everything. Lead with a column that appears in almost every `WHERE` clause.

In [ ]:
import math

ROWS = 5_000_000_000
INDEX_GRANULARITY = 8192

granules = math.ceil(ROWS / INDEX_GRANULARITY)
# One mark per granule in the primary index, plus one per granule in each key column file.
print(f"granules: {granules:,}")
print(f"marks if the index covers 3 columns: {granules * 3:,}")

assert granules == 610_352
assert granules * 3 == 1_831_056

## How do the sparse primary index and skipping indexes work?

The primary index is sparse: it stores one mark per granule of `index_granularity` rows (8192 by default) rather than one entry per row. A range scan finds the first matching granule and reads forward, which keeps the index small enough to stay in memory. Skipping indexes add secondary pruning for columns that are not in the sort key.

```sql
ALTER TABLE analytics.events
    ADD INDEX idx_type event_type TYPE set(100) GRANULARITY 4,
    ADD INDEX idx_amount amount TYPE minmax GRANULARITY 2,
    ADD INDEX idx_message message TYPE tokenbf_v1(8192, 3, 0) GRANULARITY 4;

ALTER TABLE analytics.events MATERIALIZE INDEX idx_type;

EXPLAIN indexes = 1
SELECT count() FROM analytics.events
WHERE event_type = 'purchase' AND amount > 1000;
```

The output reports which granules each index skipped:

```text
PrimaryKey  Keys: event_type, event_time, user_id   Granules: 116/1024
Skip        Name: idx_amount (minmax)               Granules: 32/116
```

Index guidance: `minmax` for numeric ranges, `set` for a bounded number of values, `bloom_filter` for equality on high-cardinality strings, and `tokenbf_v1` or `ngrambf_v1` for text search. Every index costs storage and merge time, so add one only after a query shows up in `system.query_log` as a full scan.

A mark maps a granule to a byte offset in a column file. The primary index stores marks only for the key columns, so an index lookup finds the first granule whose key can match, then reads forward until the key range ends.

Because the index is sparse, it can skip granules but not rows. A point lookup on the sort key still reads a whole granule of 8192 rows and filters inside it.

Skip indexes work on blocks of granules. `GRANULARITY n` groups n primary granules into one index block, so the index prunes at that coarser granularity. A smaller `GRANULARITY` prunes more precisely and costs more index entries.

The sparse index in miniature: DuckDB generates a table physically sorted by `ORDER BY (event_type, event_time)`, and one min/max mark per 8192-row granule decides which granules a filter must read.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE events AS
    SELECT
        ['click', 'purchase', 'signup', 'view'][1 + (i % 4)] AS event_type,
        TIMESTAMP '2026-10-01' + INTERVAL (i // 4) SECOND AS event_time,
        (hash(i) % 100000)::BIGINT AS user_id,
        round((i % 500) / 3.0, 2) AS amount
    FROM range(400000) t(i)
    ORDER BY event_type, event_time          -- the MergeTree ORDER BY: physical sort order
""")

GRANULE = 8192
marks = con.execute(f"""
    WITH numbered AS (SELECT *, (row_number() OVER () - 1) // {GRANULE} AS granule FROM events)
    SELECT granule, min(event_type) AS min_type, max(event_type) AS max_type,
           min(user_id) AS min_user, max(user_id) AS max_user
    FROM numbered GROUP BY granule ORDER BY granule
""").fetchall()


def granules_to_read(predicate) -> int:
    return sum(predicate(mark) for mark in marks)


leading = granules_to_read(lambda m: m[1] <= "purchase" <= m[2])           # WHERE event_type = 'purchase'
non_leading = granules_to_read(lambda m: m[3] <= 4242 <= m[4])             # WHERE user_id = 4242
assert len(marks) == 49
assert leading <= 14 and non_leading == len(marks)                         # leading key skips ~3/4
print(f"granules: {len(marks)} | event_type filter reads {leading} | user_id filter reads {non_leading}")

## What are projections and when should you use them?

A projection is an alternative physical ordering of the same data, stored inside the table. It lets one table serve two access patterns without a second table or a materialized view.

```sql
ALTER TABLE analytics.events
    ADD PROJECTION by_user (
        SELECT user_id, event_time, event_type, amount
        ORDER BY (user_id, event_time)
    );

ALTER TABLE analytics.events MATERIALIZE PROJECTION by_user;

EXPLAIN indexes = 1
SELECT user_id, count() FROM analytics.events WHERE user_id = 101;
```

The planner selects a projection automatically when it fits the query. Use projections for a second common sort order or for pre-aggregated rollups on the same table. They cost extra disk and merge work, so keep the number small and materialize only what queries use. For rollups that must be shared and queried by name, a `MaterializedView` over `AggregatingMergeTree` is usually clearer.

## Which MergeTree engine variants exist?

The MergeTree family changes what happens during a merge. Choosing the right variant removes the need for application-side deduplication or rollup jobs.

| Engine | Merge behavior | Use case |
|---|---|---|
| `MergeTree` | keeps all rows | append-only facts and events |
| `ReplacingMergeTree` | keeps one row per sort key | upserts and CDC mirrors |
| `SummingMergeTree` | sums numeric columns per key | additive counters |
| `AggregatingMergeTree` | merges aggregate states | exact rollups with `uniq`, `avg`, `quantile` |
| `CollapsingMergeTree` | cancels rows by `sign` | streaming updates without versions |
| `VersionedCollapsingMergeTree` | cancels by `sign` and version | out-of-order updates |

```sql
CREATE TABLE analytics.orders_rt
(
    order_id   UInt64,
    status     LowCardinality(String),
    amount     Decimal(18, 2),
    updated_at DateTime
)
ENGINE = ReplacingMergeTree(updated_at)
ORDER BY order_id;

-- SummingMergeTree follows the same shape: ENGINE = SummingMergeTree((hits, bytes))
```

`ReplacingMergeTree` deduplicates only during merges, so duplicates remain visible until a merge runs; `FINAL` or an `argMax` aggregation is required for correctness. `SummingMergeTree` keeps one row per key after merging, so a pre-merge sum can differ from the post-merge value.

## How do the Distributed and Replicated engines work?

`Replicated*MergeTree` keeps each shard's data on multiple nodes through ClickHouse Keeper. `Distributed` is a thin table that routes reads and writes across shards of a cluster.

```sql
CREATE TABLE analytics.events_local ON CLUSTER analytics_cluster
(
    event_time DateTime,
    user_id    UInt64,
    country    LowCardinality(String),
    amount     Decimal(18, 2)
)
ENGINE = ReplicatedMergeTree('/clickhouse/tables/{shard}/analytics/events', '{replica}')
PARTITION BY toYYYYMM(event_time)
ORDER BY (country, event_time);

CREATE TABLE analytics.events ON CLUSTER analytics_cluster AS analytics.events_local
ENGINE = Distributed('analytics_cluster', 'analytics', 'events_local', rand());

SELECT country, sum(amount) FROM analytics.events GROUP BY country;
```

Rules that avoid surprises: use `rand()` or a high-cardinality key for even distribution; use `GLOBAL JOIN` or `distributed_product_mode` for joins across `Distributed` tables; and write to a `Distributed` table only when you accept that the sharding key decides placement.

`ReplicatedMergeTree` coordinates replicas through ClickHouse Keeper. One replica takes the insert, writes the part locally, and records the part name in Keeper. Other replicas see the entry and fetch the part, so every replica eventually holds the same parts. If a replica falls behind, its `absolute_delay` grows and it catches up by copying parts from healthy replicas.

`Distributed` holds no data. It reads the shard list from `remote_servers`, sends the query to every shard, and merges the partial results on the initiator. For `sum(x)`, each shard returns a partial sum and the initiator adds them. For `GROUP BY`, each shard returns its groups and the initiator re-aggregates, so fan-out amplifies both latency and network traffic.

Shard design decides whether the fan-out is even. `rand()` spreads rows uniformly and makes every shard do equal work. A low-cardinality shard key creates a hot shard. A high-cardinality key such as `user_id` keeps data local for per-user queries but makes global aggregations read every shard.

## How does TTL manage the data lifecycle?

TTL statements run during merges to delete, move, or aggregate old data. Retention becomes part of the table definition instead of a batch job.

```sql
CREATE TABLE analytics.logs
(
    log_time DateTime,
    service  LowCardinality(String),
    level    LowCardinality(String),
    message  String
)
ENGINE = MergeTree()
PARTITION BY toDate(log_time)
ORDER BY (service, log_time)
TTL log_time + INTERVAL 30 DAY DELETE,
    log_time + INTERVAL 7 DAY TO VOLUME 'cold';

-- Aggregating TTL: keep hourly error rollups longer than raw rows
ALTER TABLE analytics.logs
    MODIFY TTL
        log_time + INTERVAL 7 DAY DELETE,
        log_time + INTERVAL 7 DAY GROUP BY toStartOfHour(log_time), service, level
            SET message = '', hits = count()
        WHERE level = 'error';
```

TTL is applied on merge, not on a wall clock, so a partition can briefly outlive its TTL. Pair `DELETE` TTLs with partition-level `DROP PARTITION` for hard guarantees, and use `TO VOLUME` or `TO DISK` with a storage policy that defines hot and cold tiers.

TTL rules are stored in the table metadata and evaluated while parts merge. A `DELETE` rule drops rows whose expression is in the past. A `TO VOLUME` or `TO DISK` rule moves a whole part to another storage tier once every row in the part passes the expression. A `GROUP BY` rule rewrites rows into aggregate states after a delay.

The TTL merge runs on its own schedule, controlled by the MergeTree setting `merge_with_ttl_timeout`, separate from normal merges. Because `TO VOLUME` moves a part only when all of its rows satisfy the rule, a part that spans a wide time range delays the move.

## How do materialized views do incremental aggregation?

A ClickHouse materialized view is a trigger on inserts, not a cached query result. It reads the inserted block and writes to a target table, which makes it the standard tool for incremental aggregation.

```sql
CREATE TABLE analytics.daily_country
(
    day     Date,
    country LowCardinality(String),
    users   AggregateFunction(uniq, UInt64),
    revenue SimpleAggregateFunction(sum, Decimal(18, 2)),
    events  SimpleAggregateFunction(sum, UInt64)
)
ENGINE = AggregatingMergeTree()
ORDER BY (day, country);

CREATE MATERIALIZED VIEW analytics.mv_daily_country TO analytics.daily_country AS
SELECT
    toDate(event_time) AS day,
    country,
    uniqState(user_id) AS users,
    sum(amount)        AS revenue,
    count()            AS events
FROM analytics.events
GROUP BY day, country;
```

Only rows inserted after the view is created are processed, so create the target table and view before backfilling raw data, or backfill the target explicitly. Use `AggregateFunction` for exact distinct or quantiles and `SimpleAggregateFunction` for plain `sum`, `min`, `max`, and `any`.

Why the target stores aggregate *states* (`uniqState`, merged with `uniqMerge`) rather than finished numbers: sums can be added batch by batch, distinct counts cannot. The model below fires a "view" on every insert batch:

In [ ]:
batches = [
    [("DE", 1, 10.0), ("DE", 2, 5.0), ("FR", 3, 7.0)],
    [("DE", 1, 3.0), ("FR", 3, 1.0), ("FR", 4, 2.0)],      # users 1 and 3 come back
]
revenue: dict[str, float] = {}
users_counts: dict[str, int] = {}          # wrong: finished numbers per batch
users_states: dict[str, set] = {}          # right: mergeable state (ClickHouse keeps an HLL sketch)

for batch in batches:                       # the view sees only the rows of each insert
    for country, user, amount in batch:
        revenue[country] = revenue.get(country, 0) + amount
    for country in {c for c, _, _ in batch}:
        batch_users = {u for c, u, _ in batch if c == country}
        users_counts[country] = users_counts.get(country, 0) + len(batch_users)
        users_states.setdefault(country, set()).update(batch_users)

assert revenue == {"DE": 18.0, "FR": 10.0}
assert users_counts == {"DE": 3, "FR": 3}                   # double-counts returning users
assert {c: len(s) for c, s in users_states.items()} == {"DE": 2, "FR": 2}
print("revenue:", revenue, "| summed counts:", users_counts,
      "| merged states:", {c: len(s) for c, s in users_states.items()})

## How do you ingest from Kafka with the Kafka engine?

The Kafka table engine turns a topic into a table. Reads consume messages, and a materialized view on top of it is the usual way to land data into a `MergeTree` target.

```sql
CREATE TABLE analytics.events_kafka
(
    event_time DateTime,
    user_id    UInt64,
    country    LowCardinality(String),
    amount     Decimal(18, 2)
)
ENGINE = Kafka
SETTINGS
    kafka_broker_list = 'kafka-1:9092,kafka-2:9092',
    kafka_topic_list = 'events',
    kafka_group_name = 'clickhouse-events',
    kafka_format = 'JSONEachRow',
    kafka_num_consumers = 4,
    kafka_max_block_size = 65536,
    kafka_skip_broken_messages = 100;

CREATE MATERIALIZED VIEW analytics.mv_events_kafka TO analytics.events AS
SELECT event_time, user_id, country, amount
FROM analytics.events_kafka;
```

The Kafka engine table is a cursor, not storage, so never query it for analytics directly. Duplicate consumption is normal, so make the target idempotent or use `ReplacingMergeTree`. A parse error blocks the consumer unless `kafka_skip_broken_messages` is set.

## How do async inserts, batch inserts, and Kafka Connect differ?

ClickHouse is built for bulk inserts. Row-by-row `INSERT ... VALUES` is the classic anti-pattern, because every insert creates a part and many small parts trigger merge pressure.

Async inserts let the server buffer small writes and form larger blocks, which suits many clients that each send a row at a time. Batch inserts are better when the client controls the flow.

```sql
INSERT INTO analytics.events SETTINGS
    async_insert = 1,
    wait_for_async_insert = 1,
    async_insert_max_data_size = 10485760,
    async_insert_busy_timeout_ms = 1000
VALUES (now(), 101, 'DE', 'purchase', 42.50);
```

```python
import clickhouse_connect

client = clickhouse_connect.get_client(host="ch-1", port=8123, database="analytics")

rows = [
    ("2026-10-04 10:00:00", 101, "DE", "purchase", 42.50),
    ("2026-10-04 10:00:01", 102, "FR", "view", 0.00),
]

client.insert(
    "events",
    rows,
    column_names=["event_time", "user_id", "country", "event_type", "amount"],
    settings={"async_insert": 1, "wait_for_async_insert": 1, "max_insert_threads": 4},
)
```

`max_insert_block_size` (default 1048576 rows) caps the block the server forms, `min_insert_block_size_rows` and `min_insert_block_size_bytes` control squashing before parts are written, `input_format_parallel_parsing` speeds up large file loads, and `async_insert_deduplicate` deduplicates identical buffered inserts on retries. Aim for at least 10k to 100k rows per part on a busy table.

The ClickHouse Kafka Connect sink is the alternative when you already run Kafka Connect and want exactly-once delivery and connector management in one place.

```json
{
  "name": "clickhouse-events-sink",
  "config": {
    "connector.class": "com.clickhouse.kafka.connect.ClickHouseSinkConnector",
    "tasks.max": "4",
    "topics": "events",
    "hostname": "ch-1",
    "port": "8443",
    "database": "analytics",
    "username": "ingest",
    "ssl": "true",
    "exactlyOnce": "true",
    "topic2TableMap": "events=events_raw"
  }
}
```

The Kafka engine is lower-latency and lives inside ClickHouse, but offsets and parse errors are managed by the server. Kafka Connect gives exactly-once semantics and DLQ handling at the cost of another service to operate. Both write into the same `MergeTree` targets.

An insert creates at least one part per partition it touches. A client that inserts one row per statement creates one part per row, and merges cannot keep up with that rate. Async inserts solve this on the server: rows wait in a buffer until `async_insert_busy_timeout_ms` expires or `async_insert_max_data_size` is reached, then the server writes one part.

Batch inserts solve it on the client: buffer rows and send 10k to 100k per statement. This is the fastest path and the one to prefer whenever the client controls the cadence. `min_insert_block_size_rows` and `min_insert_block_size_bytes` also squash blocks before they become parts.

## How do codecs and compression work?

ClickHouse compresses every column independently and lets you choose a codec per column. Codecs exploit the column's value pattern, which is why the columnar layout compresses far better than row storage.

```sql
CREATE TABLE analytics.metrics
(
    ts          DateTime CODEC(DoubleDelta, ZSTD(3)),
    device_id   UInt64 CODEC(Delta, ZSTD(3)),
    temperature Float32 CODEC(Gorilla, ZSTD(3)),
    status      LowCardinality(String) CODEC(ZSTD(3)),
    raw         String CODEC(NONE)
)
ENGINE = MergeTree()
PARTITION BY toDate(ts)
ORDER BY (device_id, ts);

SELECT name, formatReadableSize(sum(data_uncompressed_bytes) / sum(data_compressed_bytes)) AS ratio
FROM system.columns WHERE table = 'metrics' GROUP BY name;
```

`Delta` and `DoubleDelta` are excellent for monotonically increasing timestamps and counters, `Gorilla` for slowly changing floats, `T64` for small integer ranges, and `ZSTD` as the general-purpose backstop. Higher ZSTD levels save storage but cost insert CPU. Check the real ratio in `system.columns` rather than assuming, and leave already-compressed payloads as `NONE`.

Why `Delta`/`DoubleDelta` help: a second-resolution timestamp column delta-encodes to a run of identical small numbers, which a general-purpose compressor then squeezes far better.

In [ ]:
import struct
import zlib

timestamps = [1_791_000_000 + i for i in range(100_000)]                  # one row per second
raw = struct.pack(f"<{len(timestamps)}q", *timestamps)
deltas = [timestamps[0]] + [b - a for a, b in zip(timestamps, timestamps[1:])]
delta_encoded = struct.pack(f"<{len(deltas)}q", *deltas)

raw_ratio = len(raw) / len(zlib.compress(raw, 6))
delta_ratio = len(raw) / len(zlib.compress(delta_encoded, 6))
assert delta_ratio > 20 * raw_ratio
print(f"compression ratio, raw: {raw_ratio:.1f}x | after Delta: {delta_ratio:.0f}x")

## Which column types matter, and what are the Nullable caveats?

Type choice drives both compression and query speed. The biggest wins come from replacing generic strings with specialized types.

```sql
CREATE TABLE analytics.sessions
(
    session_id  UUID,
    user_id     UInt64,
    country     LowCardinality(String),
    device      Enum8('desktop' = 1, 'mobile' = 2, 'tablet' = 3),
    started_at  DateTime64(3, 'UTC'),
    duration_ms UInt32,
    revenue     Decimal(18, 2),
    tags        Array(LowCardinality(String)),
    props       Map(String, String)
)
ENGINE = MergeTree()
ORDER BY (country, started_at);
```

+ `LowCardinality(String)` dictionary-encodes a column with fewer than roughly 10k distinct values. It saves space and speeds up grouping and filtering, but hurts if cardinality is actually high.
+ `Enum8` or `Enum16` is cheaper than `LowCardinality(String)` when the value set is fixed and known.
+ `Array`, `Map`, and `Tuple` are first-class, and array functions avoid the row explosion that join-then-group would cause.
+ `JSON` has been production-ready since 25.3. It stores each frequent path as its own typed sub-column, so `payload.user.id` reads like an ordinary column. Pin hot paths with type hints and drop noisy ones, as in `payload JSON(user.id UInt64, SKIP debug)`; `max_dynamic_paths` (default 1024) caps how many paths become sub-columns. Prefer it to a `String` column parsed with `JSONExtract*` at query time.

`Nullable(T)` stores a separate null mask, blocks some optimizations, and cannot be used efficiently in the sort key, so prefer a default or an explicit sentinel.

```sql
-- Prefer a plain column with a default over Nullable on the hot path
CREATE TABLE analytics.events_no_null
(event_time DateTime, country LowCardinality(String) DEFAULT 'unknown')
ENGINE = MergeTree() ORDER BY (country, event_time);
```

`LowCardinality(String)` stores each distinct value once in a dictionary and keeps a small integer key per row. Filtering and grouping then compare integers, which is fast and compact. The win disappears as the number of distinct values approaches the row count, because the dictionary grows and every read pays a lookup. Measure `uniqExact` before you choose it.

`Enum8` and `Enum16` store a fixed value set as a one-byte or two-byte code with named labels, which is cheaper than `LowCardinality` when the set is known and does not change. `Array`, `Map`, and `Tuple` are first-class, and array functions avoid the row explosion a join-then-group would cause.

## How do ClickHouse JOIN algorithms work?

Joins in ClickHouse are hash-based by default and the build side is the right-hand table. Distribution, memory, and algorithm all matter more than in a row-store.

```sql
SET join_algorithm = 'hash';
SET max_bytes_in_join = 20000000000;
SET join_use_nulls = 1;

SELECT e.country, d.region, count()
FROM analytics.events AS e
LEFT JOIN analytics.dict_country AS d ON e.country = d.country
GROUP BY e.country, d.region;
```

+ `hash` (default) - builds a hash table from the right side. Fast when the right side fits in memory.
+ `grace_hash` - hash join that spills to disk; the safest choice for large joins.
+ `parallel_hash` - parallel build and probe, good for large right sides on many cores.
+ `partial_merge` and `full_sorting_merge` - sort-merge joins that avoid a full hash table.
+ `direct` - uses the right table's storage order directly, with no hash table.
+ `GLOBAL JOIN` - required when joining two `Distributed` tables so the right side is sent to each shard once.

ClickHouse reads the right table first and builds a hash table from its join key, unless the algorithm says otherwise. The right table is the build side and the left table streams through the probe. Memory is the limit, so the right table must be the small one.

`hash` keeps the whole right side in memory. `grace_hash` partitions both sides and spills partitions to disk. `parallel_hash` builds the table with several threads. `partial_merge` and `full_sorting_merge` sort both sides and merge. `direct` reads the right table in key order with no hash table, which works when the right table is small and has no duplicate keys.

## How do dictionaries speed up lookups?

Dictionaries are key-value lookups loaded into memory or partly on SSD and refreshed on a lifetime. They replace a repeated join with a function call.

```sql
CREATE DICTIONARY analytics.country_region
(
    country String,
    region  String,
    manager String
)
PRIMARY KEY country
SOURCE(CLICKHOUSE(host 'ch-1' port 9000 db 'analytics' table 'country_reference' user 'default'))
LAYOUT(HASHED())
LIFETIME(MIN 300 MAX 600);

SELECT e.country,
       dictGet('analytics.country_region', 'region', e.country) AS region,
       count() AS events
FROM analytics.events AS e
WHERE e.event_time >= today() - 1
GROUP BY e.country, region
ORDER BY events DESC;
```

Layouts: `FLAT` for small numeric keys, `HASHED` for general keys, `COMPLEX_KEY_HASHED` for composite keys, `RANGE_HASHED` for time-bounded attributes, and `CACHE` or `SSD_CACHE` when the reference data is too large for memory. An unloaded or stale dictionary shows up in `system.dictionaries` as `status = 'NOT_LOADED'` or with an old `last_successful_update_time`, so monitor it like a job.

## How do you read EXPLAIN output and optimize queries?

`EXPLAIN` has several modes, and `indexes = 1` is the one that tells you whether the query prunes data or scans everything.

```sql
EXPLAIN PIPELINE SELECT country, count() FROM analytics.events GROUP BY country;
EXPLAIN indexes = 1 SELECT count() FROM analytics.events WHERE country = 'DE';
```

Then confirm with the query log:

```sql
SELECT query_duration_ms,
       read_rows,
       read_bytes,
       memory_usage,
       ProfileEvents['SelectedMarks'] AS marks
FROM system.query_log
WHERE type = 'QueryFinish' AND query LIKE '%analytics.events%'
ORDER BY event_time DESC
LIMIT 10;
```

+ Filter on the leading `ORDER BY` column so `PrimaryKey` granules are skipped.
+ Add a skipping index only for a column that appears in slow queries.
+ Reduce `SelectedMarks`; if it equals total marks, the sort key does not match the filter.
+ Replace `SELECT *` with explicit columns, because wide tables read many column files.
+ Prefer `uniqCombined` for approximate distinct counts and `AggregatingMergeTree` for repeated rollups.
+ Use `max_threads`, `max_memory_usage`, and `max_execution_time` as per-query guardrails, tracked in `system.processes`.

```sql
-- Slowest queries in the last day, by bytes read
SELECT
    normalized_query_hash,
    any(query)                               AS sample,
    count()                                  AS runs,
    round(avg(query_duration_ms))            AS avg_ms,
    round(quantile(0.95)(query_duration_ms)) AS p95_ms,
    sum(read_rows)                           AS rows_read,
    formatReadableSize(sum(read_bytes))      AS bytes_read
FROM system.query_log
WHERE type = 'QueryFinish'
  AND event_time > now() - INTERVAL 1 DAY
  AND query_kind = 'Select'
GROUP BY normalized_query_hash
ORDER BY sum(read_bytes) DESC
LIMIT 10;
```

Common mistake: tuning by wall-clock time alone. A query that finishes in 50 ms but reads a billion rows looks fine on a small table and collapses at 100x the data. Watch `read_bytes` and `SelectedMarks` to see whether the work is proportional to the answer or to the whole table.

## How do you deduplicate with ReplacingMergeTree and FINAL?

`ReplacingMergeTree` keeps the row with the greatest version per sort key during a merge. Because merges are asynchronous, duplicates remain queryable until then, and `FINAL` resolves them at read time.

```sql
CREATE TABLE analytics.users_rt
(
    user_id    UInt64,
    plan       LowCardinality(String),
    email      String,
    updated_at DateTime,
    is_deleted UInt8 DEFAULT 0
)
ENGINE = ReplacingMergeTree(updated_at, is_deleted)
ORDER BY user_id;
```

Three ways to read the deduplicated state:

```sql
-- Correct but merges everything at query time
SELECT * FROM analytics.users_rt FINAL WHERE is_deleted = 0;

-- Faster on large tables: aggregate at query time
SELECT user_id,
       argMax(plan, updated_at) AS plan,
       argMax(email, updated_at) AS email
FROM analytics.users_rt
GROUP BY user_id
HAVING argMax(is_deleted, updated_at) = 0;

-- Force a merge now, then query without FINAL
OPTIMIZE TABLE analytics.users_rt FINAL;
```

The same three read strategies in DuckDB: raw rows (duplicates visible before a merge), the `FINAL` equivalent (latest version per key), and the `argMax` aggregation that production queries prefer.

In [ ]:
con.execute("""
    CREATE TABLE users_rt AS SELECT * FROM (VALUES
        (1, 'free', TIMESTAMP '2026-10-01 10:00', 0),
        (1, 'pro',  TIMESTAMP '2026-10-03 09:00', 0),
        (2, 'pro',  TIMESTAMP '2026-10-01 11:00', 0),
        (2, 'pro',  TIMESTAMP '2026-10-04 08:00', 1),      -- is_deleted tombstone version
        (3, 'free', TIMESTAMP '2026-10-02 12:00', 0)
    ) t(user_id, plan, updated_at, is_deleted)
""")
raw_rows = con.execute("SELECT count(*) FROM users_rt").fetchone()[0]
final = con.execute("""
    SELECT user_id, plan FROM users_rt
    QUALIFY row_number() OVER (PARTITION BY user_id ORDER BY updated_at DESC) = 1 AND is_deleted = 0
    ORDER BY user_id
""").fetchall()
arg_max = con.execute("""
    SELECT user_id, arg_max(plan, updated_at) AS plan FROM users_rt
    GROUP BY user_id HAVING arg_max(is_deleted, updated_at) = 0 ORDER BY user_id
""").fetchall()
assert raw_rows == 5
assert final == arg_max == [(1, "pro"), (3, "free")]
print("before merge:", raw_rows, "rows | deduplicated:", final)

## How do you build real-time analytics pipelines?

A real-time pipeline on ClickHouse is usually a raw table, one or more incremental materialized views, and a query layer over aggregated tables.

Patterns worth copying: keep raw data short-lived and rollups long-lived; make every stage idempotent because Kafka ingestion is at-least-once; and use `ReplacingMergeTree` for dimensions that change, with a periodic `OPTIMIZE` for readers that cannot tolerate duplicates.

Every stage of the pipeline must be idempotent because Kafka delivery is at-least-once. A raw table keyed by `(event_id)` in a `ReplacingMergeTree`, or a deduplicating insert block, absorbs replays. Rollup tables fed by `AggregatingMergeTree` states are naturally idempotent per insert block only if the source block is not duplicated, so deduplicate upstream or key the raw table.

Operate the pipeline by watching the stages. `system.parts` shows whether a rollup is growing, `system.merges` shows whether merges keep up, and `system.query_log` shows whether the views fire on every insert.

```sql
-- Raw events are deduplicated on replay by event_id
CREATE TABLE analytics.events_raw
(
    event_id   UUID,
    event_time DateTime,
    country    LowCardinality(String),
    amount     Decimal(18, 2)
)
ENGINE = ReplacingMergeTree()
PARTITION BY toDate(event_time)
ORDER BY event_id
TTL event_time + INTERVAL 3 DAY DELETE;

-- Confirm each stage is alive
SELECT table, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE database = 'analytics' AND active
  AND table IN ('events_raw', 'events_minute', 'daily_country')
GROUP BY table
ORDER BY parts DESC;
```

Common mistake: building the rollup from the raw table with `SELECT ... FROM events_raw FINAL` inside a scheduled job. That scans and merges raw data on every run. Let the materialized view update the rollup on insert, and reserve the scheduled job for a periodic repair or backfill.

## How does ClickHouse compare with Druid, BigQuery, and Snowflake?

| System | Storage and compute | Strength | Trade-off |
|---|---|---|---|
| ClickHouse | shared-nothing when self-managed; separated (object storage, SharedMergeTree) in ClickHouse Cloud | sub-second queries on high-volume event data, self-managed or cloud | operational effort, limited multi-statement transactions |
| Druid | separated, segment-based | pre-aggregated time-series with rollup at ingestion | complex ingestion specs, weaker general SQL |
| BigQuery | fully separated, serverless | elastic scale, zero operations, big ecosystem | cost per query, less control over physical layout |
| Snowflake | separated, virtual warehouses | concurrency isolation, mature governance, SQL breadth | cost, higher latency for high-frequency small queries |

```text
Choose ClickHouse for high-volume append-only events with sub-second aggregation
targets and a team that can operate a cluster; choose BigQuery or Snowflake when
operations, governance, and elasticity matter more than hardware-level control.
```

ClickHouse's closest cousin is Druid: both pre-aggregate at ingestion and serve time-series rollups. ClickHouse is closer to a general SQL database, with full SQL, joins, dictionaries, and table engines that read files directly. BigQuery and Snowflake separate storage from compute, which makes them elastic and low-maintenance. Self-managed ClickHouse couples them, which makes each node fast but makes resizing a rebalancing exercise rather than a slider; ClickHouse Cloud's SharedMergeTree removes most of that rebalancing work. Choose on the operating model and on latency targets, not only on benchmark numbers.

## What operational concerns matter in production?

ClickHouse rewards proactive operations. Three areas cause most incidents.

**Merges.** Merge throughput must keep up with insert rate, or parts accumulate and reads slow down.

```sql
SELECT database, table, elapsed, progress, num_parts FROM system.merges ORDER BY elapsed DESC;
```

**Keeper and replication.** Replication correctness depends on ClickHouse Keeper being healthy and quorum-connected.

```sql
SELECT * FROM system.replicas WHERE is_readonly OR absolute_delay > 60;
SELECT * FROM system.replication_queue ORDER BY create_time LIMIT 20;
SELECT * FROM system.zookeeper WHERE path = '/clickhouse/tables';
```

**Mutations, backups, and guardrails.** `ALTER TABLE ... UPDATE` and `ALTER TABLE ... DELETE` are asynchronous mutations that rewrite whole columns in every affected part; keep them for retention and backfills. For row-level corrections use the lightweight statements. `DELETE FROM` marks rows in a hidden `_row_exists` mask, and later merges remove them. `UPDATE ... SET`, generally available since 26.10, writes small patch parts holding only the changed rows and columns; `SELECT` applies them at once and merges fold them in later. Both still cost more than inserts, so use them for occasional fixes, not OLTP-style churn.

```sql
ALTER TABLE analytics.events DELETE WHERE event_time < today() - 365;
-- Lightweight row-level fixes; UPDATE needs the table settings
-- enable_block_number_column = 1 and enable_block_offset_column = 1
DELETE FROM analytics.events WHERE user_id = 42;
UPDATE analytics.events SET country = 'DE' WHERE user_id = 43 AND country = 'XX';
SELECT * FROM system.mutations WHERE is_done = 0;
BACKUP TABLE analytics.events TO Disk('backups', 'events.zip');
```

## How do you design the ORDER BY and primary key from real filters?

Start from the queries, not from the schema. List the columns each query filters on, and rank them by how often they appear. The leading `ORDER BY` column must be one that almost every query constrains, because the sparse index can only skip when a filter restricts a prefix of the key.

Order the key as: equality filters first, then the time column, then high-cardinality columns used for ordering. A filter on a prefix of `ORDER BY` prunes granules. A filter on a later column alone does not.

```sql
-- Query patterns to design for:
--   1. WHERE country = ? AND event_time >= ?   -> (country, event_time)
--   2. WHERE user_id = ?                       -> projection on (user_id)
--   3. WHERE event_type = ?                    -> skip index or nothing

CREATE TABLE analytics.events_design
(
    event_time DateTime,
    country    LowCardinality(String),
    user_id    UInt64,
    event_type LowCardinality(String)
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(event_time)
ORDER BY (country, event_time, user_id)
SETTINGS index_granularity = 8192;

EXPLAIN indexes = 1
SELECT count() FROM analytics.events_design
WHERE country = 'DE' AND event_time >= '2026-10-01';
```

Then confirm the choice against production traffic in `system.query_log`, and revisit it when the query mix changes.

Common mistake: leading with the most selective column. If `user_id` is nearly unique, `ORDER BY (user_id, event_time)` looks attractive, but a query that filters only time cannot use the index and scans every granule. Matching the common filter prefix beats raw selectivity. If two sort orders are both common, keep one as `ORDER BY` and add the other as a projection.

## How do you batch inserts and keep the number of parts low?

Part count is the main write-side health metric. Every insert creates a part per partition it touches. Merges combine parts, but a merge costs far more CPU per byte than an insert, so the insert rate must stay under the merge throughput. The cheapest control is to send fewer, larger inserts.

```sql
-- Parts per partition right now
SELECT table, partition, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE active AND database = 'analytics'
GROUP BY table, partition
ORDER BY parts DESC
LIMIT 20;

-- Thresholds that first delay, then reject inserts
SELECT name, value, changed, description
FROM system.merge_tree_settings
WHERE name IN ('parts_to_delay_insert', 'parts_to_throw_insert', 'max_parts_in_total');
```

Target 10k to 100k rows per insert on a busy table, and let async inserts buffer many small writers into one part.

Common mistake: one `INSERT` per event from an application. Each call becomes a part, merges fall behind, and `parts_to_delay_insert` starts throttling the writer. Buffer in the client, or set `async_insert = 1` with `wait_for_async_insert = 1` for clients that cannot batch. Do not raise the delay and throw thresholds to work around the backlog; fix the insert pattern instead.

## How do you choose codecs, LowCardinality, and skipping structures?

Storage choices compound: a codec shrinks a column, `LowCardinality` shrinks a string column's keys, and a projection or skip index trades storage for fewer scanned granules. Choose each one from the column's value pattern, then measure the real ratio in `system.columns`.

```sql
CREATE TABLE analytics.metrics_tuned
(
    ts        DateTime CODEC(DoubleDelta, ZSTD(1)),
    device_id UInt64   CODEC(Delta, ZSTD(1)),
    status    LowCardinality(String),
    value     Float64  CODEC(Gorilla, ZSTD(1)),
    payload   String   CODEC(ZSTD(3))
)
ENGINE = MergeTree()
ORDER BY (device_id, ts);

-- Measure compression per column
SELECT name,
       formatReadableSize(sum(data_compressed_bytes)) AS compressed,
       round(sum(data_uncompressed_bytes) / sum(data_compressed_bytes), 2) AS ratio
FROM system.columns
WHERE table = 'metrics_tuned'
GROUP BY name
ORDER BY sum(data_compressed_bytes) DESC;
```

Use `Delta` or `DoubleDelta` for timestamps and counters, `T64` for small integer ranges, `Gorilla` for slowly changing floats, and `ZSTD` as the general backstop.

## How do you tune JOINs and distributed query fan-out?

A join builds a hash table from the right table, so the right side must be the small one. Pick the algorithm by whether that table fits in memory: `hash` when it does, `grace_hash` to spill when it does not, `direct` when the right table is tiny and keyed.

For distributed queries, the initiator sends the query to every shard and merges the results. Latency is close to the slowest shard, and network traffic grows with the number of shards times the partial result each one returns. A shard key of `rand()` spreads rows evenly; a low-cardinality shard key creates a hot shard.

```sql
SET join_algorithm = 'grace_hash';
SET max_bytes_in_join = 20000000000;

SELECT c.region, count() AS events
FROM analytics.events AS e
LEFT JOIN analytics.country_ref AS c ON e.country = c.country
GROUP BY c.region
ORDER BY events DESC;

-- Fan-out cost seen from the initiator
SELECT query, read_rows, result_rows, memory_usage
FROM system.query_log
WHERE type = 'QueryFinish' AND query LIKE '%JOIN%'
ORDER BY read_rows DESC
LIMIT 5;
```

Keep lookup tables in dictionaries when the same join repeats, because a dictionary lookup is a hash call instead of a join.

Common mistake: joining two `Distributed` tables without `GLOBAL`. Each shard then builds a hash table from only its own slice of the right table, so rows on different shards never match. Use `GLOBAL JOIN`, or set `distributed_product_mode = 'global'` when cross-shard joins are unavoidable. Prefer denormalizing the dimension into the fact table instead.

## Real world use cases for ClickHouse

These patterns recur in production. Each one leans on a different part of the engine: incremental materialized views for freshness, TTL and storage policies for retention, `ReplacingMergeTree` for mutable entities, and table functions for reading object storage in place.

### 1. Real-time product analytics

A consumer app streams click and purchase events through Kafka. A materialized view keeps per-minute rollups, and dashboards query the rollup table with sub-second latency while raw events stay for three days.

```sql
SELECT toStartOfMinute(event_time) AS minute,
       country,
       uniqMerge(users) AS users,
       sum(revenue)     AS revenue
FROM analytics.events_minute
WHERE minute >= now() - INTERVAL '6' HOUR
  AND country = 'DE'
GROUP BY minute
ORDER BY minute;
```

Why ClickHouse: incremental materialized views mean the dashboard never scans raw events, and the columnar engine aggregates millions of rows per query without a pre-computation job.

### 2. Observability and log analytics with retention tiers

A platform stores structured application logs, keeps 7 days on fast SSD and 90 days on cheaper storage, and searches messages with a token index.

```sql
CREATE TABLE observability.logs
(
    log_time DateTime,
    service  LowCardinality(String),
    level    LowCardinality(String),
    message  String CODEC(ZSTD(3)),
    INDEX idx_message message TYPE tokenbf_v1(8192, 3, 0) GRANULARITY 4
)
ENGINE = MergeTree()
PARTITION BY toDate(log_time)
ORDER BY (service, level, log_time)
TTL log_time + INTERVAL 7 DAY TO VOLUME 'hot',
    log_time + INTERVAL 90 DAY DELETE;
```

Why ClickHouse: TTL storage policies implement retention without a batch job, and skipping indexes keep free-text queries inside interactive latency.

### 3. CDC mirror of an operational database

Debezium events land in a `ReplacingMergeTree` keyed by primary key. A periodic `OPTIMIZE` keeps the table compact, and readers that need a fully deduplicated view use `FINAL` or `argMax` as described above.

```sql
CREATE TABLE cdc.customers_rt
(
    customer_id UInt64,
    name        String,
    email       String,
    updated_at  DateTime,
    is_deleted  UInt8 DEFAULT 0
)
ENGINE = ReplacingMergeTree(updated_at, is_deleted)
ORDER BY customer_id;
```

Why ClickHouse: `ReplacingMergeTree` gives last-write-wins semantics without a merge-on-read table format, and the columnar layout still answers analytical questions over the mirrored data.

### 4. Querying a lakehouse in place

A data team keeps Parquet and Iceberg tables in S3. Instead of loading everything, they query the lake directly with the `iceberg` and `s3` table functions from the same cluster that serves dashboards.

```sql
SELECT country, count() AS events, sum(amount) AS revenue
FROM iceberg('s3://acme-warehouse/analytics/events/', 'access_key', 'secret_key')
WHERE event_date = '2026-10-04'
GROUP BY country
ORDER BY revenue DESC;
```

Why ClickHouse: the same engine that serves real-time dashboards can scan object storage without a separate ingestion pipeline, which shortens the path from raw files to an answer.

The same query-in-place idea runs locally with DuckDB: write a Parquet file, then aggregate it without loading it into a table first.

In [ ]:
import pathlib
import tempfile

lake = pathlib.Path(tempfile.mkdtemp()) / "events.parquet"
con.execute(f"COPY (SELECT * FROM events) TO '{lake}' (FORMAT parquet)")
by_type = con.execute(f"""
    SELECT event_type, count(*) AS events, round(sum(amount), 2) AS revenue
    FROM read_parquet('{lake}') GROUP BY event_type ORDER BY event_type
""").fetchall()
assert [row[0] for row in by_type] == ["click", "purchase", "signup", "view"]
assert sum(row[1] for row in by_type) == 400_000
print(by_type)

### 5. Multi-tenant usage metering and billing

A SaaS platform meters API calls per customer per hour, sums counters with `SummingMergeTree`, and reconciles invoices with the same rollup table.

```sql
SELECT tenant_id,
       sum(calls) AS calls,
       round(sum(errors) / nullIf(sum(calls), 0), 4) AS error_rate
FROM billing.usage_hourly
WHERE hour >= toStartOfMonth(now())
GROUP BY tenant_id
ORDER BY calls DESC;
```

Why ClickHouse: `SummingMergeTree` keeps additive counters compact and merge-friendly, and the same rollup serves both live usage dashboards and month-end reconciliation.

## ClickHouse exercises

The numbers are estimates, not promises. Real results depend on types, cardinality, sort order, codec, and cluster shape; the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Choose an ORDER BY key from query patterns

**Task.** Three of four query patterns filter on `country` and `event_time`. Choose the leading `ORDER BY` columns from the columns common to all patterns, ordered by cardinality.

**Hint.** Intersect the `where` lists, then sort the shared columns by cardinality so the lowest-cardinality column leads.

**Expected result.** `("country", "event_time")`.

In [ ]:
QUERY_PATTERNS = [
    {"where": ["country", "event_time"]},
    {"where": ["country", "event_time", "user_id"]},
    {"where": ["country", "event_time"]},
]
CARDINALITY = {"country": 200, "event_time": 500_000_000, "user_id": 50_000_000}

def choose_order_by(patterns, cardinality):
    common = set(patterns[0]["where"])
    for pattern in patterns[1:]:
        common &= set(pattern["where"])
    return tuple(sorted(common, key=lambda column: cardinality[column]))

key = choose_order_by(QUERY_PATTERNS, CARDINALITY)
print("ORDER BY", key)
print("leading column appears in every pattern:", all("country" in p["where"] for p in QUERY_PATTERNS))

assert key == ("country", "event_time")
assert key[0] == "country"

### Try it 2. Compute granule count from rows and index_granularity

**Task.** For 5 billion rows and `index_granularity = 8192`, compute the number of granules and the number of marks if the primary index covers three key columns.

**Hint.** `granules = ceil(rows / index_granularity)`; marks are one per granule per indexed column.

**Expected result.** 610,352 granules and 1,831,056 marks.

In [ ]:
import math

ROWS = 5_000_000_000
INDEX_GRANULARITY = 8192
KEY_COLUMNS = 3

granules = math.ceil(ROWS / INDEX_GRANULARITY)
marks = granules * KEY_COLUMNS
print(f"granules: {granules:,}")
print(f"marks for {KEY_COLUMNS} key columns: {marks:,}")

assert granules == 610_352
assert marks == 1_831_056

### Try it 3. Decide skip index versus projection

**Task.** Return the right structure for each workload: a bounded filter column, a second frequent sort order, or a high-cardinality column that is never filtered alone.

**Hint.** A second sort order needs a projection; a bounded filter column needs a skip index; otherwise add nothing.

**Expected result.** `skip index`, `projection`, `no extra index`.

In [ ]:
MAX_SKIP_INDEX_DISTINCT = 10_000

def choose_structure(distinct_values, filtered_alone, needs_second_sort_order):
    if needs_second_sort_order:
        return "projection"
    if filtered_alone and distinct_values <= MAX_SKIP_INDEX_DISTINCT:
        return "skip index"
    return "no extra index"

cases = [
    (500, True, False),
    (50_000_000, True, True),
    (50_000_000, False, False),
]
for case in cases:
    print(case, "->", choose_structure(*case))

assert choose_structure(500, True, False) == "skip index"
assert choose_structure(50_000_000, True, True) == "projection"
assert choose_structure(50_000_000, False, False) == "no extra index"

### Try it 4. Estimate parts growth from insert cadence

**Task.** A writer sends 120 single-row inserts per minute. Merge capacity is 1.5 parts per second. Compute parts per day, merge capacity per day, and parts per day if the writer batches 1,000 inserts into one.

**Hint.** `parts_per_day = inserts_per_minute * 1440`; batching divides it by the batch factor.

**Expected result.** 172,800 parts, 129,600 merges of capacity, 173 parts when batched.

In [ ]:
import math

INSERTS_PER_MINUTE = 120
MERGE_PARTS_PER_SECOND = 1.5
BATCH_FACTOR = 1_000
MINUTES_PER_DAY = 24 * 60

parts_per_day = INSERTS_PER_MINUTE * MINUTES_PER_DAY
merge_capacity_per_day = int(MERGE_PARTS_PER_SECOND * 86_400)
parts_if_batched = math.ceil(parts_per_day / BATCH_FACTOR)

print(f"parts per day: {parts_per_day:,}")
print(f"merge capacity per day: {merge_capacity_per_day:,}")
print(f"parts per day when batched: {parts_if_batched:,}")

assert parts_per_day == 172_800
assert parts_per_day > merge_capacity_per_day
assert parts_if_batched == 173

### Try it 5. Pick a codec by column type

**Task.** Choose a codec for a timestamp, a float gauge, and an already-compressed blob.

**Hint.** Delta-family codecs for ordered values, `Gorilla` for slowly changing floats, `NONE` for data that is already compressed.

**Expected result.** `DoubleDelta, ZSTD(1)`, `Gorilla, ZSTD(1)`, `NONE`.

In [ ]:
def choose_codec(column_type, monotonic, is_float, already_compressed):
    if already_compressed:
        return "NONE"
    if column_type == "DateTime" or monotonic:
        return "DoubleDelta, ZSTD(1)"
    if is_float:
        return "Gorilla, ZSTD(1)"
    return "ZSTD(1)"

for args in [
    ("DateTime", True, False, False),
    ("Float64", False, True, False),
    ("String", False, False, True),
]:
    print(args, "->", choose_codec(*args))

assert choose_codec("DateTime", True, False, False) == "DoubleDelta, ZSTD(1)"
assert choose_codec("Float64", False, True, False) == "Gorilla, ZSTD(1)"
assert choose_codec("String", False, False, True) == "NONE"

### Try it 6. Compute compression ratio from read_bytes

**Task.** A table holds 1,200 GiB uncompressed and 150 GiB on disk. Compute the compression ratio and the space saved.

**Hint.** `ratio = uncompressed / compressed`; saved is the difference.

**Expected result.** Ratio 8.0, saving 1,050 GiB.

In [ ]:
UNCOMPRESSED_GIB = 1_200
COMPRESSED_GIB = 150

ratio = UNCOMPRESSED_GIB / COMPRESSED_GIB
saved_gib = UNCOMPRESSED_GIB - COMPRESSED_GIB
print(f"ratio: {ratio:.1f}x")
print(f"saved: {saved_gib:,} GiB")

assert ratio == 8.0
assert saved_gib == 1_050

### Try it 7. Model distributed fan-out cost

**Task.** A single node scans a table in 2.0 seconds. Model an 8-shard cluster that splits the rows evenly, then compute the partial rows returned to the initiator when each shard returns 10,000 groups.

**Hint.** Each shard scans `rows / shards`; the initiator receives `shards * partial_rows` and re-aggregates them.

**Expected result.** 500,000,000 rows per shard, 0.25 s ideal scan time, 80,000 partial rows at the initiator.

In [ ]:
SHARDS = 8
ROWS_TOTAL = 4_000_000_000
SINGLE_NODE_SECONDS = 2.0
PARTIAL_GROUPS_PER_SHARD = 10_000

rows_per_shard = ROWS_TOTAL / SHARDS
ideal_scan_seconds = SINGLE_NODE_SECONDS / SHARDS
rows_to_initiator = SHARDS * PARTIAL_GROUPS_PER_SHARD

print(f"rows per shard: {rows_per_shard:,.0f}")
print(f"ideal scan time: {ideal_scan_seconds:.2f} s")
print(f"partial rows at initiator: {rows_to_initiator:,}")

assert rows_per_shard == 500_000_000
assert ideal_scan_seconds == 0.25
assert rows_to_initiator == 80_000

### Try it 8. Check whether a hash join fits in memory

**Task.** The right table has 20 million rows at 120 bytes each. The memory limit is 2 GiB. Decide between `hash` and `grace_hash`.

**Hint.** Build size is `rows * bytes_per_row`; compare it with the limit.

**Expected result.** 2.4 GB does not fit under 2 GiB, so choose `grace_hash`.

In [ ]:
RIGHT_ROWS = 20_000_000
BYTES_PER_ROW = 120
MAX_BYTES_IN_JOIN = 2 * 1024**3

build_bytes = RIGHT_ROWS * BYTES_PER_ROW
fits = build_bytes <= MAX_BYTES_IN_JOIN
algorithm = "hash" if fits else "grace_hash"

print(f"build side: {build_bytes / 1e9:.1f} GB")
print(f"limit:      {MAX_BYTES_IN_JOIN / 1e9:.2f} GB")
print("algorithm:", algorithm)
print("better fix: put the smaller table on the right")

assert build_bytes == 2_400_000_000
assert not fits
assert algorithm == "grace_hash"